# Syllable generation (fast)

Same outputs as `5_syllable_generation.ipynb`, about 10x faster. The code is in `syllable_pipeline.py`.

**What it makes**, for one dataset:
1. **states files**: one per session, every frame with its identifiable state (movement x whisker x lick),
   its trial variables and its trial-phase label;
2. **syllables**: per trial and trial phase, the state sequence binned to `TARGET_LENGTH`;
3. **trials**: one row per trial with its trial-level variables;
4. **raw** (optional, large): the same binning for licks, whisker and paw velocities.

**The choices** (everything else, from folders to file names, follows from them and is printed below):
- `DATASET`: `proficient` (2 cameras), `neuromodulators`, or a training stage (`session_1`, `last_training`,
  `biased`; 1 camera);
- `MOVEMENT` and `K`: the FIRST digit of a syllable and its number of states. `both_paws` (2-camera data only),
  `left_paw`, or `left_paw_wheel`; `K` picks `most_likely_states_<K>_*` in that clustering's folder (3.3 run with
  that `VAR` and `K`);
- `WHEEL` and `K_WHEEL`: add the wheel's own states (3.3 with `VAR = 'wheel'`) as a LAST digit. Not with
  `left_paw_wheel`, which has the wheel inside already;
- `STATE_SPACE`: which clustering made the states. `own` = 3.3 fitted on this dataset;
  `stages` = 3.3's stage-balanced fit (`FIT_NAME='stages'`, proficient + training);
- `LICK_HMM`: `auto` = the 100 ms fit for 1-camera data, the standard one for proficient.

A syllable is movement, whisker, lick[, wheel]: the string puts the digits side by side (with `-` between them
once a k exceeds 10), and the integer code counts through them with the movement fastest, as the original did.
Combinations whose states do not exist are refused with the list of those that do. Each states folder records
the configuration that filled it (`_config.json`) and refuses a different one, so files never get mixed.
`(proficient, both_paws, K = 8, no wheel, own)` writes to the historical names (`data/states_files/`, `8_k_10_bin_syllables_*`)
and reproduces them exactly; every other run writes to `states_files_<movement>[+wheel]_k<K>[-<K_WHEEL>]_<state space>_lick-<fit>/`.

In [23]:
DATASET = 'neuromodulators'      # 'proficient' | 'neuromodulators' | 'session_1' | 'last_training' | 'biased'
MOVEMENT = 'left_paw_wheel'      # 'both_paws' (proficient only) | 'left_paw' | 'left_paw_wheel'
K = 8                       # movement states
WHEEL = False               # True: the wheel's own states as an extra digit (not with left_paw_wheel)
K_WHEEL = 8                 # wheel states, if WHEEL
STATE_SPACE = 'own'         # 'own' (fitted on this dataset) | 'stages' (stage-balanced fit)
LICK_HMM = 'auto'           # 'auto' | 'standard' | '100ms'

STEPS = ['states', 'syllables', 'trials']   # add 'raw' for the binned raw signals (~300 MB)
STEPS = ['states', 'syllables']   # add 'raw' for the binned raw signals (~300 MB)
TARGET_LENGTH = 10          # bins per trial phase (syllables and raw)
OVERWRITE = False           # True: recompute states files that already exist
N_JOBS = 4                  # sessions in parallel (under 1 GB of memory each)

In [24]:
import importlib
import warnings
import pandas as pd
import syllable_pipeline as sp
importlib.reload(sp)
warnings.filterwarnings('ignore', category=FutureWarning)

print('clustering states on disk:\n' + sp.available().to_string(index=False))
cfg = sp.resolve(DATASET, MOVEMENT, STATE_SPACE, LICK_HMM, k=K, wheel=WHEEL, k_wheel=K_WHEEL)
print('\n' + sp.describe(cfg))
sessions = sp.sessions_with_inputs(cfg)
print(f"\n{len(sp.all_sessions(cfg))} design matrices, {len(sessions)} with all their states")

clustering states on disk:
        dataset   segmentation state_space   k
     proficient            paw         own [8]
     proficient left_paw_wheel      stages [8]
neuromodulators left_paw_wheel         own [8]
      session_1 left_paw_wheel      stages [8]
  last_training left_paw_wheel      stages [8]
         biased left_paw_wheel      stages [8]

dataset           neuromodulators
movement          left_paw_wheel
k                 8
wheel             False
k_wheel           None
state_space       own
lick_hmm          100ms
relabel           None
design_matrices   /home/ines/repositories/representation_learning_variability/paper-individuality/data/design_matrices/1_camera_setup/NM/
wavelets          /home/ines/repositories/representation_learning_variability/paper-individuality/data/neuromodulators/left_paw_wavelets/
states_dir        /home/ines/repositories/representation_learning_variability/paper-individuality/data/neuromodulators/left_paw_wheel_states_uniform_raw_sessionz/
w

## 1. States files

In [25]:
if 'states' in STEPS:
    print(sp.run_states(cfg, sessions, n_jobs=N_JOBS, overwrite=OVERWRITE))

/home/ines/miniconda3/envs/iblenv/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


{'written': 533}


## 2. Syllables

In [26]:
if 'syllables' in STEPS:
    syllables = sp.run_syllables(cfg, sessions, TARGET_LENGTH, n_jobs=N_JOBS)
    path = sp.output_name(cfg, 'syllables', TARGET_LENGTH)
    syllables.to_parquet(path, compression='gzip')
    print(f'{len(syllables)} rows -> {path}')

1586597 rows -> /home/ines/repositories/representation_learning_variability/paper-individuality/data/neuromodulators/left_paw_wheel_k8_own_lick-100ms_neuromodulators_8_k_10_bin_syllables_06-10-2026


## 3. Trials
Every design matrix with a states file. (The original appended the previous session's trials again when a
session had no states file; `all_trials_10-09-2026` has SWC_053 sixteen times and 8 sessions twice.)

In [20]:
if 'trials' in STEPS:
    all_trials = sp.run_trials(cfg, sp.all_sessions(cfg))
    path = sp.output_name(cfg, 'trials')
    all_trials.to_parquet(path, compression='gzip')
    print(f"{len(all_trials)} trials, {all_trials['session'].nunique()} sessions -> {path}")

## 4. Raw sequences (optional)

In [ ]:
if 'raw' in STEPS:
    raw = sp.run_raw(cfg, sp.all_sessions(cfg), TARGET_LENGTH, n_jobs=N_JOBS)
    path = sp.output_name(cfg, 'raw', TARGET_LENGTH)
    raw.to_parquet(path, compression='gzip')
    print(f'{len(raw)} rows -> {path}')

## Differences from `5_syllable_generation.ipynb`
- **Identical output** (checked frame by frame / sequence by sequence): proficient states files, syllables,
  trials and raw sequences; and the old training `states_files_paw_wheel` (proficient-fitted states, 100 ms
  lick HMM).
- **Trials**: sessions without a states file are skipped instead of repeating the previous session.
- **Not carried over**: the exclusion lists (in the original they never took effect: the filter compared
  (mouse, eid) pairs with eids); the whisker/lick-only codes (cell 11) and the old per-k relabellings (k = 6, 7, 9, 10); the wheel
  syllable binning (it needed an `lda_df` that the notebook never defined).
- **Wheel digit**: the wheel states are matched to the movement frames on `Bin` (the original merged on `Bin`
  and kept the wheel file's frames); the frames, and the code numbering, are otherwise as in its cell 7 / 9.
